# UTAUT–GenAI Analysis Pipeline for EDUCON 2026

This notebook provides the **definitive, documented, and repository-ready** analysis workflow used in the study on generative AI adoption in engineering education. It is structured for **transparent reuse, local adaptation, and low-cost replication** in Google Colab or equivalent Python environments.

**Core properties**
- Nonprobability, voluntary sample
- Unweighted estimation
- No finite-population corrections
- HC3 heteroskedasticity-robust inference
- Transparent Python-based workflow


## Authorship, Terms of Use, and Citation

This notebook is part of the methodological package associated with the study:

**Duque Becerra, C. R., & Ostos Morales, I.**  
*Decision-Ready Analytic Framework for Generative AI Adoption in Engineering: Segment-Aware, Context-Sensitive, Replication-Ready.*

### Authorship
This notebook and its accompanying methodological materials were prepared by **Camilo Rene Duque Becerra** and **Isvett Ostos Morales** as part of the research and dissemination framework associated with the above study.

### Scope of use
This notebook is shared to support **academic reuse, methodological adaptation, replication, and local implementation** in engineering education and related research contexts, particularly in resource-heterogeneous settings.

Users may read, adapt, and execute this notebook for scholarly, educational, and non-commercial research purposes, provided that such use remains consistent with applicable repository licensing terms, institutional requirements, and ethical standards.

### Conditions of reuse
Reuse of this notebook does **not** imply transfer of authorship, nor does it authorize the removal of attribution to the original authors.

If this notebook, its analytic structure, its workflow logic, or its associated methodological framework inform a research study, institutional diagnostic process, adaptation, derivative workflow, or academic output, the original work **must be appropriately acknowledged and cited**.

### Citation requirement
Any substantive reuse, adaptation, or implementation of this notebook or its associated framework should include formal citation of the corresponding scholarly work and, where appropriate, of the repository in which these materials are hosted.

Users are responsible for ensuring that citation practices remain consistent with academic integrity standards and with the conventions of their target publication venue or institutional context.

### Data protection and ethical responsibility
This notebook is intended for methodological reuse.  
It does **not** grant permission to disclose confidential, identifiable, or ethically restricted data.

Users remain solely responsible for:
- obtaining any required ethics approval,
- protecting participant confidentiality,
- complying with institutional and legal data protection requirements,
- and ensuring that any datasets used with this notebook are handled responsibly.

### No warranty
This material is provided **as is**, for academic and research use, without any express or implied warranty regarding fitness for a particular purpose, completeness, legal sufficiency, or suitability for a specific institutional setting.

### Recommended citation
Please cite the associated paper when using or adapting this notebook:

**Duque Becerra, Camilo Rene, & Ostos Morales, Isvett**  
*Decision-Ready Analytic Framework for Generative AI Adoption in Engineering: Segment-Aware, Context-Sensitive, Replication-Ready.*

If a repository citation file is provided, users should also consult the repository-level citation guidance.

## Authorship and citation

**Authors**
- Camilo Rene Duque Becerra  - camilo.duque@tec.mx
- Isvett Ostos Morales       - isvettostos@tec.mx

**Affiliation**  
School of Engineering and Science, Tecnológico de Monterrey, León, Mexico

**Associated study**  
*Decision-Ready Analytic Framework for Generative AI Adoption in Engineering: Segment-Aware, Context-Sensitive, Replication-Ready.*

**Suggested use**
This notebook is intended for methodological reuse and local adaptation. When this workflow informs a study design, adaptation, or analysis, please cite the associated paper and repository.

**Repository role**
This notebook is part of the reusable package that also includes:
- instrument structure
- data templates
- table templates
- adaptation guidance
- citation information


## Notebook organization

The workflow is organized into the following stages:

1. Configuration and column mapping  
2. Data loading and preprocessing  
3. Helper functions for descriptive statistics  
4. Construct-level findings  
5. Item-level asymmetries  
6. Regression and diagnostics  
7. Cluster segmentation and usage patterns  
8. Main pipeline execution  
9. Usage example for Colab

The original analytical logic has been preserved. This version improves **documentation, structure, repository readiness, and traceability**.


In [ ]:
"""
UTAUT–GenAI Analysis Pipeline for EDUCON 2026
---------------------------------------------

This script implements a fully reproducible analysis pipeline for a UTAUT-based
survey on Generative AI adoption in engineering education. It is designed to be
run in Google Colab (or any Python 3.11+ environment) and aligns with the
methodology described in the manuscript:

  4.1  Construct-level findings (unweighted; no FPC; standard & HC3 SEs)
  4.2  Item-level asymmetries (means, SD, CI, HC3 for key items)
  4.3  Regression and diagnostics (OLS with HC3, R² / adj. R², VIF, tests)
  4.4  Cluster-based segmentation and usage patterns (by activity & semester)

Key methodological properties:
  - Nonprobability, voluntary sample
  - Unweighted estimation
  - No finite-population corrections (FPC)
  - HC3 heteroskedasticity-robust standard errors for OLS
  - Transparent, open, low-cost toolchain (pandas, numpy, statsmodels, sklearn)

Author: Authors: Duque B., Camilo R and Ostos M., Isvett Y.
"""

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, normal_ad
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# =============================================================================

## Configuration: column mapping

Define the exact mapping between raw survey headers and analytic constructs. Adapt only if your local dataset uses different column names.

In [ ]:
# 0. CONFIGURATION: COLUMN MAP (ADAPT TO YOUR EXACT HEADERS IF NEEDED)
# =============================================================================

# Mapping from raw Spanish column headers to analytical constructs/roles.
# These strings must match your .xlsx header row exactly. If there are small
# differences (e.g., extra spaces, typos), adjust them here.
COLUMN_MAP = {
    "PE": [
        "Creo que el uso de la IAG mejorará mi rendimiento académico.",
        "Cre que la IAG me ayudará a comprender mejor los conceptos de mis asignaturas.",
        "Creo que incorporar la IAG aumentará la eficiencia en la realización de mis tareas y proyectos.",
        "Creo que el uso de la IAG facilitará mi preparación para exámenes y evaluaciones.",
    ],
    "EE": [
        "Creo que Aprender a utilizar la IAG resulta sencillo para mí.",
        "Creo que La interacción con la IAG es clara y comprensible. ",
        "Creo que Integrar la IAG en mis actividades académicas requiere un esfuerzo mínimo.",
        "Estoy seguro/a de que podré dominar el uso de la IAG sin dificultades.",
    ],
    "SI": [
        "Creo que Mis compañeros apoyan el uso de la IAG en el aprendizaje.",
        "Creo que Mis profesores fomentan activamente el uso de la IAG. ",
        "Creo que La comunidad educativa valora positivamente la implementación de la IAG. ",
        "Creo que Las personas importantes para mí consideran que debo utilizar la IAG en mis estudios. ",
    ],
    "FC": [
        "Creo que La institución educativa proporciona los recursos tecnológicos necesarios para utilizar la IAG. ",
        "Creo que Tengo acceso a formación y capacitación sobre el uso de la IAG. ",
        "Creo que Existe soporte técnico disponible para resolver problemas relacionados con la IAG. ",
        "Creo que Las políticas y directrices de mi institución favorecen la integración de la IAG en el aprendizaje.",
    ],
    "age": "Edad:",
    "gender": "Género:",
    "semester": "Semestre cursando actualmente de mi programa académico",
    "major": "Área de estudio de mi programa",
    "prior_use": "¿Has tenido experiencia previa utilizando tecnologías de inteligencia artificial?",
    "freq_search": "¿Con qué frecuencia utilizas la IAG para buscar información o investigar temas académicos?",
    "freq_summaries": "¿Con qué frecuencia usas la IAG para realizar resúmenes o para la elaboración de tareas?",
    "freq_projects": "¿Con qué frecuencia integras la IAG en el desarrollo de proyectos o trabajos de curso?",
    "freq_exams": "¿Con qué frecuencia recurres a la IAG para prepararte para exámenes o estudiar para evaluaciones?",
}


# =============================================================================

## Data loading and preprocessing

Load the Excel dataset, coerce numeric fields, derive construct scores, and prepare the main covariates used downstream.

In [ ]:
# 1. DATA LOADING AND PREPROCESSING
# =============================================================================

def load_data(file_path: str) -> pd.DataFrame:
    """
    Load the Excel dataset.

    Parameters
    ----------
    file_path : str
        Path to the .xlsx file uploaded in Colab (e.g., "respuestas.xlsx").

    Returns
    -------
    df : pandas.DataFrame
        Raw dataframe as read from Excel.
    """
    df = pd.read_excel(file_path)
    return df


def clean_and_derive(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean raw data, derive construct scores and BI proxy, and prepare
    basic covariates.

    Notes
    -----
    - All analyses are unweighted; no finite-population corrections.
    - Numeric columns are coerced to numeric; missing values are allowed.
    - A simple mean-imputation is applied for numeric variables when needed
      in regression and clustering to avoid dropping rows. This should be
      documented in the Methods/Limitations section of the manuscript.

    Returns
    -------
    df_out : pandas.DataFrame
        Dataframe with additional columns: PE, EE, SI, FC, BI, semester_num,
        prior_use_bin, gender_cat.
    """
    df = df.copy()

    # Coerce Likert and frequency items to numeric
    all_likert_cols = (
        COLUMN_MAP["PE"] +
        COLUMN_MAP["EE"] +
        COLUMN_MAP["SI"] +
        COLUMN_MAP["FC"]
    )
    freq_cols = [
        COLUMN_MAP["freq_search"],
        COLUMN_MAP["freq_summaries"],
        COLUMN_MAP["freq_projects"],
        COLUMN_MAP["freq_exams"],
    ]

    for col in all_likert_cols + freq_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Construct scores = row-wise mean (ignoring NaNs)
    df["PE"] = df[COLUMN_MAP["PE"]].mean(axis=1)
    df["EE"] = df[COLUMN_MAP["EE"]].mean(axis=1)
    df["SI"] = df[COLUMN_MAP["SI"]].mean(axis=1)
    df["FC"] = df[COLUMN_MAP["FC"]].mean(axis=1)

    # Behavioral Intention proxy: mean of frequency items
    df["BI"] = df[freq_cols].mean(axis=1)

    # Semester: extract numeric part (e.g., "4to semestre" -> 4)
    sem_raw = df[COLUMN_MAP["semester"]].astype(str)
    df["semester_num"] = (
        sem_raw.str.extract(r"(\d+)", expand=False).astype(float)
    )

    # Prior use: map to binary (1 = Yes, 0 = No), robust to accents
    prior = df[COLUMN_MAP["prior_use"]].astype(str).str.strip().str.lower()
    df["prior_use_bin"] = np.where(prior.str.startswith("s"), 1.0,
                            np.where(prior.str.startswith("y"), 1.0, 0.0))

    # Gender: keep as categorical string
    df["gender_cat"] = df[COLUMN_MAP["gender"]].astype(str).str.strip()

    return df


# =============================================================================

## Helper functions: regression export compatibility

This helper cell is retained to preserve the regression table export logic used by the original analytical workflow.

In [ ]:
def format_p(p):
    return "<0.001" if p < 1e-3 else f"{p:.3f}"

def build_regression_table(model, term_order=None, out_prefix="table_ols_regression",
                           required_terms=("const", "prior_use_bin")):
    """
    Robusta a modelos ajustados con numpy o pandas.
    - Obtiene nombres de términos desde model.model.exog_names.
    - Si no existen, genera nombres de respaldo.
    - Exporta full_precision y paper con columna 'Term'.
    - Incluye aserciones de coherencia.
    """
    # Resultados robustos HC3
    m = model.get_robustcov_results(cov_type="HC3")

    # === Nombres de términos (robusto) ===
    try:
        term_names = list(getattr(m, "model").exog_names)
    except Exception:
        # Respaldo: si no hay nombres, genera x0..x{p-1}
        p = len(m.params)
        term_names = [f"x{i}" for i in range(p)]

    # Sanidad: longitud debe coincidir
    if len(term_names) != len(m.params):
        raise ValueError(
            f"Length mismatch: exog_names({len(term_names)}) vs params({len(m.params)}). "
            "Ensure the model was fit with the final design matrix."
        )

    # Base con precisión completa
    reg = pd.DataFrame({
        "Term": term_names,
        "Coef": np.asarray(m.params, dtype=float),
        "SE_HC3": np.asarray(m.bse, dtype=float),
        "t": np.asarray(m.tvalues, dtype=float),
        "p": np.asarray(m.pvalues, dtype=float),
    })

    # IC 95%
    ci = m.conf_int(alpha=0.05)
    # conf_int puede devolver ndarray (n,2) o DataFrame; forzamos a ndarray
    ci = np.asarray(ci)
    reg["CI95_L"] = ci[:, 0]
    reg["CI95_U"] = ci[:, 1]

    # ---- Aserciones de coherencia (opcionales pero recomendadas) ----
    # Solo valido si esperas esos términos; si no, comenta el bloque
    missing = set(required_terms) - set(reg["Term"])
    if missing:
        # En vez de abortar, damos advertencia clara:
        print(f"[WARN] Missing expected terms in regression table: {missing}. "
              f"Available terms: {reg['Term'].tolist()}")

    if "const" in reg["Term"].values and "prior_use_bin" in reg["Term"].values:
        row_const = reg.loc[reg["Term"] == "const", ["Coef","SE_HC3","CI95_L","CI95_U"]].values[0]
        row_prior = reg.loc[reg["Term"] == "prior_use_bin", ["Coef","SE_HC3","CI95_L","CI95_U"]].values[0]
        # Tolerancia muy estricta: si llega a dispararse, es porque realmente son idénticos
        if np.allclose(row_const, row_prior, atol=1e-12, rtol=0.0):
            print("[WARN] Intercept and prior_use_bin look numerically identical; "
                  "check the design matrix and variable names.")

    # ---- Orden estable de filas ----
    if term_order is not None:
        known = [t for t in term_order if t in reg["Term"].values]
        extras = [t for t in reg["Term"] if t not in known]
        reg = reg.set_index("Term").loc[known + extras].reset_index()

    # ---- Export 1: máxima precisión (auditoría) ----
    reg.to_csv(f"{out_prefix}_full_precision.csv", index=False)

    # ---- Versión “paper”: redondeos y p formateada ----
    def _format_p(p):
        try:
            return "<0.001" if float(p) < 1e-3 else f"{float(p):.3f}"
        except Exception:
            return str(p)

    reg_paper = reg.copy()
    reg_paper["Coef"]   = reg_paper["Coef"].round(3)
    reg_paper["SE_HC3"] = reg_paper["SE_HC3"].round(3)
    reg_paper["t"]      = reg_paper["t"].round(2)
    reg_paper["p"]      = reg_paper["p"].apply(_format_p)
    reg_paper["CI95_L"] = reg_paper["CI95_L"].round(3)
    reg_paper["CI95_U"] = reg_paper["CI95_U"].round(3)

    reg_paper.to_csv(f"{out_prefix}_paper.csv", index=False)
    return reg, reg_paper



## Helper functions: descriptives and robust standard errors

These functions compute means, confidence intervals, and HC3-based robust standard errors for intercept-only models.

In [ ]:
# 2. HELPER FUNCTIONS: DESCRIPTIVES, CI, HC3 FOR MEANS
# =============================================================================

def mean_ci(series: pd.Series, alpha: float = 0.05) -> Tuple[float, float, float]:
    """
    Compute mean and t-based 95% CI (standard formula, unweighted, no FPC).

    Parameters
    ----------
    series : pandas.Series
        Numeric data (NaNs are ignored).
    alpha : float
        Significance level (default 0.05 for 95% CI).

    Returns
    -------
    mean : float
    ci_low : float
    ci_high : float
    """
    s = series.dropna().astype(float)
    n = s.shape[0]
    if n == 0:
        return np.nan, np.nan, np.nan
    m = s.mean()
    se = s.std(ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    ci_low = m - tcrit * se
    ci_high = m + tcrit * se
    return m, ci_low, ci_high


def hc3_se_for_mean(series: pd.Series) -> float:
    """
    Compute HC3-robust standard error for the mean via intercept-only OLS.

    Parameters
    ----------
    series : pandas.Series
        Numeric data (NaNs are ignored).

    Returns
    -------
    se_hc3 : float
        HC3 robust standard error of the intercept (mean).
    """
    s = series.dropna().astype(float)
    n = s.shape[0]
    if n == 0:
        return np.nan
    y = s.values
    X = np.ones((n, 1))
    model = sm.OLS(y, X).fit(cov_type="HC3")
    se_hc3 = model.bse[0]
    return se_hc3


# =============================================================================

## Analysis 4.1: construct-level findings

Generate construct-level descriptives for PE, EE, SI, FC, and BI.

In [ ]:
# 3. ANALYSIS 4.1: CONSTRUCT-LEVEL FINDINGS
# =============================================================================

def construct_level_descriptives(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute construct-level descriptives for PE, EE, SI, FC, BI.

    Returns a DataFrame with:
        n, Mean, SD, SE, CI95_L, CI95_U, SE_HC3
    """
    constructs = ["PE", "EE", "SI", "FC", "BI"]
    rows = []

    for c in constructs:
        s = df[c].dropna().astype(float)
        n = s.shape[0]
        if n == 0:
            rows.append({
                "Construct": c,
                "n": 0,
                "Mean": np.nan,
                "SD": np.nan,
                "SE": np.nan,
                "CI95_L": np.nan,
                "CI95_U": np.nan,
                "SE_HC3": np.nan,
            })
            continue

        m = s.mean()
        sd = s.std(ddof=1)
        se = sd / np.sqrt(n)
        tcrit = stats.t.ppf(0.975, df=n - 1)
        ci_low = m - tcrit * se
        ci_high = m + tcrit * se
        se_hc3 = hc3_se_for_mean(s)

        rows.append({
            "Construct": c,
            "n": n,
            "Mean": m,
            "SD": sd,
            "SE": se,
            "CI95_L": ci_low,
            "CI95_U": ci_high,
            "SE_HC3": se_hc3,
        })

    result = pd.DataFrame(rows)
    return result


# =============================================================================

## Analysis 4.2: item-level asymmetries

Produce item-level descriptives to identify internal asymmetries within constructs, especially Effort Expectancy and Social Influence.

In [ ]:
# 4. ANALYSIS 4.2: ITEM-LEVEL ASYMMETRIES
# =============================================================================

def item_level_descriptives(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute item-level descriptives for all UTAUT items.

    Returns a DataFrame with:
        Construct, Item, n, Mean, SD, SE, SE_HC3, CI95_L, CI95_U
    """
    rows = []
    for construct, cols in [
        ("PE", COLUMN_MAP["PE"]),
        ("EE", COLUMN_MAP["EE"]),
        ("SI", COLUMN_MAP["SI"]),
        ("FC", COLUMN_MAP["FC"]),
    ]:
        for item_col in cols:
            s = df[item_col].dropna().astype(float)
            n = s.shape[0]
            if n == 0:
                rows.append({
                    "Construct": construct,
                    "Item": item_col,
                    "n": 0,
                    "Mean": np.nan,
                    "SD": np.nan,
                    "SE": np.nan,
                    "SE_HC3": np.nan,
                    "CI95_L": np.nan,
                    "CI95_U": np.nan,
                })
                continue

            m = s.mean()
            sd = s.std(ddof=1)
            se = sd / np.sqrt(n)
            tcrit = stats.t.ppf(0.975, df=n - 1)
            ci_low = m - tcrit * se
            ci_high = m + tcrit * se
            se_hc3 = hc3_se_for_mean(s)

            rows.append({
                "Construct": construct,
                "Item": item_col,
                "n": n,
                "Mean": m,
                "SD": sd,
                "SE": se,
                "SE_HC3": se_hc3,
                "CI95_L": ci_low,
                "CI95_U": ci_high,
            })

    result = pd.DataFrame(rows)
    return result


# =============================================================================

## Analysis 4.3: regression and diagnostics

Estimate the OLS model with HC3-robust standard errors and compute key diagnostics, including VIF and heteroskedasticity tests.

In [ ]:
# 5. ANALYSIS 4.3: REGRESSION AND DIAGNOSTICS
# =============================================================================

from dataclasses import dataclass
from typing import Dict
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, normal_ad
from scipy import stats

@dataclass
class RegressionResults:
    table: pd.DataFrame
    r2: float
    r2_adj: float
    vif: pd.DataFrame
    bp_test: Dict[str, float]
    jb_test: Dict[str, float]


def run_regression_and_diagnostics(df: pd.DataFrame) -> RegressionResults:
    """
    Run OLS regression BI ~ PE + EE + SI + FC + controls with HC3-robust SE via build_regression_table.
    Controls:
        - gender_cat (categorical; reference is the first category)
        - semester_num (numeric)
        - prior_use_bin (0/1)
    All predictors are coerced to float; rows with missing BI or predictors are dropped.
    """

    # 1) Selección y coerción
    vars_numeric = ["PE", "EE", "SI", "FC", "BI", "semester_num", "prior_use_bin"]
    df_model = df[vars_numeric + ["gender_cat"]].copy()

    for col in vars_numeric:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

    df_model = df_model.dropna(subset=["BI"])

    # 2) Matriz de diseño con nombres (mantener pandas!)
    X_base = df_model[["PE", "EE", "SI", "FC", "semester_num", "prior_use_bin"]]

    gender_dummies = pd.get_dummies(
        df_model["gender_cat"].astype(str).str.strip(),
        prefix="gender",
        drop_first=True,
    )

    X = pd.concat([X_base, gender_dummies], axis=1)
    X = X.apply(pd.to_numeric, errors="coerce").astype(float)
    df_reg = pd.concat([df_model[["BI"]], X], axis=1).dropna(axis=0)

    y = df_reg["BI"].astype(float)          # Series con nombre
    X = df_reg.drop(columns=["BI"])         # DataFrame con nombres
    X = sm.add_constant(X, has_constant="add")  # añade 'const' con nombre

    # 3) Ajuste OLS (sin cov_type aquí; HC3 lo aplica build_regression_table)
    model = sm.OLS(y, X, missing="drop").fit()

    # 4) Tabla OLS robusta y exportación “a prueba de Excel”
    term_order = [
        "const",                # Intercept
        "PE", "EE", "SI", "FC", # UTAUT
        "semester_num", "prior_use_bin",  # Controles
        "gender_Masculino", "gender_Otro" # Dummies (ajusta si tus etiquetas difieren)
    ]

    # Si tu variable de experiencia previa tiene otro nombre, cámbialo en required_terms y en term_order
    reg_full, reg_paper = build_regression_table(
        model,
        term_order=term_order,
        out_prefix="table_ols_regression",
        required_terms=("const", "prior_use_bin")
    )

    # 5) VIF (excluyendo la constante)
    # Usa el mismo X con nombres; elimina 'const' para el cálculo
    X_no_const = X.drop(columns=["const"])
    vif_vals = []
    for i in range(X_no_const.shape[1]):
        vif_vals.append(variance_inflation_factor(X_no_const.values, i))
    vif_df = pd.DataFrame({"Variable": X_no_const.columns, "VIF": vif_vals})

    # 6) Diagnósticos básicos
    # Residuales y exógenas del modelo (ajuste clásico; la inferencia robusta ya se aplica en la exportación)
    bp_stat, bp_pvalue, _, _ = het_breuschpagan(model.resid, model.model.exog)
    bp_test = {"stat": float(bp_stat), "pvalue": float(bp_pvalue)}

    jb_stat, jb_pvalue = normal_ad(model.resid)
    jb_test = {"stat": float(jb_stat), "pvalue": float(jb_pvalue)}

    # 7) Devuelve la tabla “full precision” para el objeto de resultados
    reg_table = reg_full.copy()  # ya contiene Term, Coef, SE_HC3, t, p, CI95_L, CI95_U

    return RegressionResults(
        table=reg_table,
        r2=float(model.rsquared),
        r2_adj=float(model.rsquared_adj),
        vif=vif_df,
        bp_test=bp_test,
        jb_test=jb_test,
    )




# =============================================================================

## Analysis 4.4: cluster segmentation and usage patterns

Generate segment-based summaries and descriptive usage patterns by academic activity and semester.

In [ ]:
# 6. ANALYSIS 4.4: CLUSTER SEGMENTATION & USAGE PATTERNS
# =============================================================================

def run_cluster_analysis(df: pd.DataFrame, n_clusters: int = 3) -> pd.DataFrame:
    """
    Run k-means clustering on standardized PE, EE, SI, FC, BI.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataframe with construct scores.
    n_clusters : int
        Number of clusters (default 3, as in the manuscript: Enthusiasts,
        Conditional Adopters, Skeptics).

    Returns
    -------
    cluster_summary : pandas.DataFrame
        Cluster-level mean scores and sizes.
    """
    df_clust = df[["PE", "EE", "SI", "FC", "BI"]].copy()
    df_clust = df_clust.apply(pd.to_numeric, errors="coerce")
    df_clust = df_clust.dropna()  # clusters on complete cases

    scaler = StandardScaler()
    Z = scaler.fit_transform(df_clust)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(Z)

    df_clust = df_clust.assign(cluster=labels)
    # Summary by cluster
    summary = df_clust.groupby("cluster").agg(
        n=("PE", "size"),
        PE_mean=("PE", "mean"),
        EE_mean=("EE", "mean"),
        SI_mean=("SI", "mean"),
        FC_mean=("FC", "mean"),
        BI_mean=("BI", "mean"),
    ).reset_index()

    return summary


def usage_by_activity(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute usage-frequency descriptives by academic activity (overall).

    Returns a DataFrame similar to the one used in the manuscript:
        Activity, n, Mean, Median, SD
    """
    freq_cols = {
        "Search & Research": COLUMN_MAP["freq_search"],
        "Summaries & Assignment Prep": COLUMN_MAP["freq_summaries"],
        "Project & Coursework Development": COLUMN_MAP["freq_projects"],
        "Exam & Evaluation Preparation": COLUMN_MAP["freq_exams"],
    }

    rows = []
    for label, col in freq_cols.items():
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        n = s.shape[0]
        if n == 0:
            rows.append({"Activity": label, "n": 0, "Mean": np.nan,
                         "Median": np.nan, "SD": np.nan})
            continue
        rows.append({
            "Activity": label,
            "n": n,
            "Mean": s.mean(),
            "Median": s.median(),
            "SD": s.std(ddof=1),
        })

    return pd.DataFrame(rows)


def usage_by_semester(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute usage-frequency descriptives by semester and activity.

    Returns a long-format DataFrame with:
        semester_num, Activity, Mean, SD, n
    """
    freq_cols = {
        "Search & Research": COLUMN_MAP["freq_search"],
        "Summaries & Assignments": COLUMN_MAP["freq_summaries"],
        "Projects & Coursework": COLUMN_MAP["freq_projects"],
        "Exam Preparation": COLUMN_MAP["freq_exams"],
    }

    df_use = df.copy()
    df_use["semester_num"] = pd.to_numeric(df_use["semester_num"], errors="coerce")

    rows = []
    for sem, sub in df_use.groupby("semester_num"):
        if pd.isna(sem):
            continue
        for label, col in freq_cols.items():
            s = pd.to_numeric(sub[col], errors="coerce").dropna()
            n = s.shape[0]
            if n == 0:
                continue
            rows.append({
                "Semester": int(sem),
                "Activity": label,
                "Mean": s.mean(),
                "SD": s.std(ddof=1),
                "n": n,
            })

    return pd.DataFrame(rows)


# =============================================================================

## Main pipeline

Run the full workflow, print the main tables, and export CSV outputs for manuscript or repository use.

In [ ]:
# 7. MAIN PIPELINE
# =============================================================================

def run_pipeline(file_path: str) -> Dict[str, object]:
    """
    High-level pipeline that executes all four analysis components:

        4.1 Construct-level findings
        4.2 Item-level asymmetries
        4.3 Regression and diagnostics
        4.4 Cluster-based segmentation and usage patterns

    Parameters
    ----------
    file_path : str
        Path to the uploaded Excel file (e.g., "respuestas.xlsx").

    Returns
    -------
    results : dict
        Dictionary with all major tables/objects:
            - "construct_descriptives"
            - "item_descriptives"
            - "regression"
            - "cluster_summary"
            - "usage_activity"
            - "usage_semester"
    """
    # Load and prepare data
    df_raw = load_data(file_path)
    df = clean_and_derive(df_raw)

    # 4.1 Construct-level findings
    construct_desc = construct_level_descriptives(df)
    print("\n[Table] Construct-level descriptives (unweighted; no FPC).")
    print(construct_desc)

    # 4.2 Item-level asymmetries
    item_desc = item_level_descriptives(df)
    print("\n[Table] Item-level descriptives (unweighted; no FPC).")
    print(item_desc)

    # 4.3 Regression and diagnostics
    reg_results = run_regression_and_diagnostics(df)
    print("\n[Table] OLS regression on BI (HC3-robust SE; unweighted).")
    print(reg_results.table)
    print(f"\nR^2 = {reg_results.r2:.3f} | Adjusted R^2 = {reg_results.r2_adj:.3f}")
    print("\n[Table] Variance Inflation Factors (VIF).")
    print(reg_results.vif)
    print("\n[Diagnostic] Breusch–Pagan test for heteroskedasticity:")
    print(reg_results.bp_test)
    print("\n[Diagnostic] Normality (Anderson–Darling) of residuals:")
    print(reg_results.jb_test)

    # 4.4 Cluster-based segmentation and usage patterns
    cluster_summary = run_cluster_analysis(df)
    print("\n[Table] Cluster-based segmentation summary (PE, EE, SI, FC, BI).")
    print(cluster_summary)

    usage_act = usage_by_activity(df)
    print("\n[Table] Usage frequency by academic activity.")
    print(usage_act)

    usage_sem = usage_by_semester(df)
    print("\n[Table] Usage frequency by semester and activity.")
    print(usage_sem)

    # Optionally, save CSVs for direct inclusion / LaTeX conversion
    construct_desc.to_csv("table_construct_descriptives.csv", index=False)
    item_desc.to_csv("table_item_descriptives.csv", index=False)
    reg_results.table.to_csv("table_ols_regression.csv", index=True)
    reg_results.vif.to_csv("table_vif.csv", index=False)
    cluster_summary.to_csv("table_cluster_summary.csv", index=False)
    usage_act.to_csv("table_usage_by_activity.csv", index=False)
    usage_sem.to_csv("table_usage_by_semester.csv", index=False)

    return {
        "construct_descriptives": construct_desc,
        "item_descriptives": item_desc,
        "regression": reg_results,
        "cluster_summary": cluster_summary,
        "usage_activity": usage_act,
        "usage_semester": usage_sem,
    }


# =============================================================================

## Usage example for Colab

The example below shows how to execute the full pipeline once the Excel dataset has been uploaded to the Colab session.

In [ ]:
# 8. USAGE EXAMPLE (FOR COLAB)
# =============================================================================
if __name__ == "__main__":
    """
    Example for running in Google Colab:

        1. Upload your Excel file (e.g., respuestas.xlsx) via the Colab UI
           or using:

               from google.colab import files
               uploaded = files.upload()

        2. Ensure COLUMN_MAP matches your exact column headers.
        3. Call:

               results = run_pipeline("respuestas.xlsx")

        4. The script will print key tables to the console and write .csv
           files to the working directory for inclusion in the manuscript.
    """
    # Example call (uncomment and adjust file name as needed):
    results = run_pipeline("respuestas.xlsx")
    pass

## Citation

If you use or adapt this notebook or its associated framework, please cite the corresponding scholarly work and consult the repository citation record (`CITATION.cff`) for updated reference details.

**Duque Becerra, Camilo, & Ostos Morales, Isvett**  
*Decision-Ready Analytic Framework for Generative AI Adoption in Engineering: Segment-Aware, Context-Sensitive, Replication-Ready.*